<a id="topo"></a>
# Atividade 2.7 — Simulador de Cache de Mapeamento Direto

**Docente:** Claudomiro de Souza Sales Junior  
**Discente:** Ana Beatriz Monteiro da Silva  
**Curso:** Inteligência Artificial


<a id="indice"></a>
## Índice

- [Atividade 2.7 (enunciado)](#atividade-27)
- [Objetivo e princípio de funcionamento](#objetivo)
- [Desenho do slide 41](#desenho)
- [Divisão do endereço e fórmulas](#formulas)
- [Código comentado](#a27-codigo)
- [Testes](#a27-testes)
- [Conclusão](#a27-conclusao)


<a id="atividade-27"></a>
## Atividade 2.7

Transformar em código o **slide 41 do PDF "Memória Cache"**, usando como base o
simulador de RAM da Atividade 2.1, mas adaptando a memória ao exemplo do slide:

- memória principal com **32 endereços de 5 bits**;
- cada endereço guarda **8 bits**;
- **blocos com 2 bytes**;
- **cache com 4 linhas**;
- endereço dividido em **TAG (2 bits), LINHA (2 bits) e BYTE (1 bit)**;
- **mapeamento direto**;
- indicar **acerto (hit)**, **falta (miss)** e **substituição por conflito**.

O programa deve permitir: (1) escrever um dado na memória RAM; (2) ler um endereço
passando pela cache; (3) listar a memória RAM; (4) mostrar TAG, validade e dados das
quatro linhas da cache; (5) encerrar.

Política de escrita pedida: **escrita direta (write-through)** — ao escrever na RAM,
atualizar também a cache **somente se aquele bloco já estiver armazenado nela**.

Usar apenas Python básico de primeiro semestre: listas, variáveis, `input`, `print`,
`if/elif/else`, `while` e `for`. Sem funções, classes, dicionários, bibliotecas ou
códigos sofisticados.


<a id="objetivo"></a>
## Objetivo e princípio de funcionamento

**Visão geral primeiro.** A memória cache é uma memória pequena e rápida que guarda
*cópias* de pedaços da memória principal (RAM) que foram acessados recentemente. Quando
a UCP pede um endereço, ela primeiro procura na cache. Se o dado já estiver lá, é um
**acerto (hit)** e a resposta é rápida. Se não estiver, é uma **falta (miss)**: o bloco
inteiro é buscado na RAM e copiado para a cache antes de devolver o dado.

**Mapeamento direto.** A RAM é dividida em *blocos* de tamanho fixo (aqui, 2 bytes cada).
No mapeamento direto, cada bloco só pode ocupar **uma única linha** da cache, calculada por:

> linha da cache = (número do bloco) **resto** (quantidade de linhas)

Como a posição é fixa, a busca é simples e barata: basta olhar **uma** linha e comparar a
**tag**. A desvantagem é o **conflito**: dois blocos diferentes que caem na mesma linha
disputam o mesmo lugar, e um expulsa o outro (**substituição por conflito**), mesmo que o
resto da cache esteja vazio.

**Como cada linha sabe qual bloco está guardando?** Cada linha guarda, além dos dados,
um **bit de validade** (1 = a linha tem um bloco válido) e uma **tag** (os bits altos do
endereço que identificam qual bloco, entre todos os que mapeiam naquela linha, está ali).
No acesso, comparamos a tag procurada com a tag guardada: iguais e válida → acerto;
diferentes ou inválida → falta.

**Neste exemplo (slide 41):**

- RAM: 32 endereços (0 a 31), endereço de **5 bits**, cada palavra com **8 bits**.
- Bloco = 2 bytes → existem **16 blocos** (0 a 15), identificados por **4 bits**.
- Cache com **4 linhas** → o número da linha ocupa **2 bits**.
- Sobram **2 bits** para a tag. O bit de mais baixa ordem (1 bit) escolhe o byte dentro
  do bloco.


<a id="desenho"></a>
## Desenho do slide 41

**INSERIR DESENHO DO SLIDE 41 AQUI**

*Descrição do que o slide mostra (para orientar a inserção da figura):* à esquerda, a
cache com 4 linhas (Linha 0 a Linha 3), cada uma com uma coluna **Tag** e uma coluna
**Dados**. Abaixo, o endereço dividido em três campos — **Tag (2 bits)**, **Linha
(2 bits)** e **Byte (1 bit)** — que alimentam o *Decodificador de linha* e o
**Comparador**, cuja saída indica **Acerto / Falta**. À direita, a memória principal
com os 32 endereços (00000 a 11111) agrupados de dois em dois, formando os 16 blocos
(0000 a 1111); por exemplo, o bloco 0001 reúne os endereços 00010 (2) e 00011 (3).


<a id="formulas"></a>
## Divisão do endereço e fórmulas

O endereço tem **5 bits**. Da esquerda para a direita:

```
 b4 b3 | b2 b1 | b0
 -----   -----   --
  TAG    LINHA  BYTE
 2 bits  2 bits 1 bit
```

| Campo | Bits | Significado | Fórmula a partir do endereço |
|-------|------|-------------|------------------------------|
| BYTE (deslocamento) | 1 | qual dos 2 bytes do bloco | `byte = endereco % 2` |
| LINHA | 2 | linha da cache (0 a 3) | `linha = (endereco // 2) % 4` |
| TAG | 2 | identifica o bloco na linha | `tag = endereco // 8` |
| (bloco) | 4 | número do bloco (0 a 15) | `bloco = endereco // 2` |

**Conferência das unidades / potências de 2:**

- Quantidade de endereços: 2⁵ = 32 endereços → endereço de 5 bits. ✔
- Tamanho do bloco: 2 bytes = 2¹ → 1 bit de deslocamento de byte. ✔
- Número de blocos: 32 bytes ÷ 2 bytes/bloco = 16 = 2⁴ → 4 bits de bloco. ✔
- Número de linhas: 4 = 2² → 2 bits de linha. ✔
- Tag: 4 bits do bloco − 2 bits de linha = 2 bits de tag. ✔

**Exemplo (endereço 10):** 10 = `01010`. TAG = `01` (1), LINHA = `01` (1), BYTE = `0`.
Bloco = 10 // 2 = 5 = `0101`, e de fato tag·4 + linha = 1·4 + 1 = 5. ✔


<a id="a27-codigo"></a>
## Código comentado

O programa é dividido em células pequenas: primeiro preparamos a **RAM**, depois a
**cache**, depois o **roteiro automático de comandos desta atividade** e,
por fim, o **programa principal** com o menu de 5 opções.


### 1) Memória RAM (32 endereços de 8 bits)

A RAM é uma lista com 32 posições. Cada posição guarda uma palavra de 8 bits, ou seja,
um número inteiro de 0 a 255 (2⁸ = 256 valores possíveis). Começamos tudo com zero,
igual ao simulador da Atividade 2.1.


In [1]:
# cria a RAM como uma lista de 32 posicoes, todas com 0
ram = []
contador = 0
while contador < 32:
    ram.append(0)            # cada palavra comeca zerada (00000000)
    contador = contador + 1

print("RAM criada com", len(ram), "enderecos.")
print("Conteudo inicial (tudo zero):")
print(ram)

RAM criada com 32 enderecos.
Conteudo inicial (tudo zero):
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


### 2) A cache (4 linhas)

Como não podemos usar dicionários nem classes, representamos a cache com **quatro listas
paralelas**, todas com 4 posições (uma por linha). Para a linha `i`:

- `cache_validade[i]` → 1 se a linha guarda um bloco válido, 0 se está vazia;
- `cache_tag[i]` → a tag do bloco guardado (usamos -1 quando a linha está vazia);
- `cache_byte0[i]` → o primeiro byte do bloco (deslocamento 0);
- `cache_byte1[i]` → o segundo byte do bloco (deslocamento 1).

Assim cada linha tem, como pede o enunciado, **bit de validade, tag e os dois bytes** do
bloco.


In [2]:
# quatro listas paralelas, uma posicao para cada linha (0 a 3)
cache_validade = [0, 0, 0, 0]      # 0 = linha vazia, 1 = linha valida
cache_tag      = [-1, -1, -1, -1]  # -1 indica que ainda nao ha bloco
cache_byte0    = [0, 0, 0, 0]      # primeiro byte do bloco (deslocamento 0)
cache_byte1    = [0, 0, 0, 0]      # segundo byte do bloco (deslocamento 1)

print("Cache criada com 4 linhas, todas vazias no inicio.")

Cache criada com 4 linhas, todas vazias no inicio.


### 3) Roteiro automático de comandos

Nesta Atividade 2.7, a lista `comandos` representa uma sequência de operações já preparada.
Cada item usa o formato `[opcao, endereco, valor]`. Isso permite executar e conferir todos os
testes sem digitação durante essa parte do notebook.

Opções do roteiro:

- **1** = escrever um valor na RAM (usa `endereco` e `valor`);
- **2** = ler um endereço passando pela cache (usa `endereco`);
- **3** = listar a RAM;
- **4** = mostrar a cache;
- **5** = encerrar.

A sequência abaixo foi escolhida **só para teste** e cobre todos os casos exigidos
(miss compulsório, hit, outro byte do mesmo bloco, substituição por conflito, miss por
conflito, limites de endereço e as duas situações da escrita direta).


In [3]:
# roteiro de comandos: cada linha eh [opcao, endereco, valor]
# (opcoes 3, 4 e 5 nao usam endereco/valor; deixamos 0)
comandos = [
    [1, 2, 170],   # escreve 170 (10101010) no endereco 2  -> so dado de teste
    [1, 3, 85],    # escreve 85  (01010101) no endereco 3  -> so dado de teste
    [2, 2, 0],     # le o endereco 2  -> esperado: MISS compulsorio
    [2, 2, 0],     # le o endereco 2  -> esperado: HIT
    [2, 3, 0],     # le o endereco 3  -> esperado: HIT (outro byte do mesmo bloco)
    [2, 10, 0],    # le o endereco 10 -> esperado: MISS por conflito (substituicao)
    [2, 2, 0],     # le o endereco 2  -> esperado: MISS por conflito (bloco antigo saiu)
    [2, 0, 0],     # le o endereco 0  -> limite inferior, MISS
    [2, 31, 0],    # le o endereco 31 -> limite superior, MISS
    [1, 2, 200],   # escreve 200 no endereco 2  -> bloco esta na cache: atualiza os dois
    [1, 10, 123],  # escreve 123 no endereco 10 -> bloco ausente: atualiza so a RAM
    [3, 0, 0],     # lista a RAM
    [4, 0, 0],     # mostra a cache
    [5, 0, 0],     # encerra
]

print("Roteiro com", len(comandos), "comandos pronto.")

Roteiro com 14 comandos pronto.


### 4) Programa principal (o simulador)

Este é o coração da atividade. Para cada comando do roteiro, o programa:

1. separa o endereço em **tag, linha e byte** com as fórmulas já vistas;
2. na **leitura**, compara a tag procurada com a tag guardada na linha e decide
   **acerto** ou **falta**; em caso de falta, traz o bloco da RAM e avisa se houve
   **substituição por conflito**;
3. na **escrita**, grava na RAM e, pela política de **escrita direta**, só atualiza a
   cache se aquele bloco já estiver guardado nela;
4. ao final, mostra as **estatísticas** (leituras, acertos, faltas, substituições e taxa
   de acerto).


In [4]:
# ===================================================================
# SIMULADOR DE CACHE DE MAPEAMENTO DIRETO
# Em um terminal, cada valor viria de input(); aqui vem do roteiro "comandos".
# ===================================================================

# contadores para as estatisticas
total_leituras = 0
total_acertos = 0
total_faltas = 0
total_substituicoes = 0

for comando in comandos:
    opcao = comando[0]
    endereco = comando[1]
    valor = comando[2]

    # ----------------------------------------------------------------
    # OPCAO 1: ESCREVER NA RAM (escrita direta / write-through)
    # ----------------------------------------------------------------
    if opcao == 1:
        if endereco < 0 or endereco > 31:
            print("Endereco invalido:", endereco, "(precisa estar entre 0 e 31)")
        else:
            print("--- ESCRITA ---")
            print("Gravando o valor", valor, "no endereco", endereco)
            # 1) escrita direta: a RAM eh sempre atualizada
            ram[endereco] = valor
            # 2) descobre bloco, byte, linha e tag desse endereco
            bloco = endereco // 2
            byte = endereco % 2
            linha = bloco % 4
            tag = bloco // 4
            # 3) so mexe na cache se aquele bloco ja estiver guardado nela
            if cache_validade[linha] == 1 and cache_tag[linha] == tag:
                if byte == 0:
                    cache_byte0[linha] = valor
                else:
                    cache_byte1[linha] = valor
                print("O bloco ja estava na cache (linha", linha, ") -> RAM e cache atualizadas.")
            else:
                print("O bloco nao estava na cache -> atualizei apenas a RAM.")
            print()

    # ----------------------------------------------------------------
    # OPCAO 2: LER UM ENDERECO PASSANDO PELA CACHE
    # ----------------------------------------------------------------
    elif opcao == 2:
        if endereco < 0 or endereco > 31:
            print("Endereco invalido:", endereco, "(precisa estar entre 0 e 31)")
        else:
            total_leituras = total_leituras + 1
            # separa o endereco nos tres campos
            bloco = endereco // 2
            byte = endereco % 2
            linha = bloco % 4
            tag = bloco // 4
            # monta a parte binaria de cada campo (so para mostrar bonito)
            tag_bin = str(tag // 2) + str(tag % 2)        # 2 bits
            linha_bin = str(linha // 2) + str(linha % 2)  # 2 bits
            byte_bin = str(byte)                          # 1 bit
            endereco_bin = tag_bin + linha_bin + byte_bin # 5 bits

            print("--- LEITURA ---")
            print("Endereco", endereco, "=", endereco_bin,
                  "( TAG", tag_bin, "| LINHA", linha_bin, "| BYTE", byte_bin, ")")
            print("Bloco:", bloco, " | Linha:", linha,
                  " | Tag procurada:", tag, " | Deslocamento:", byte)

            # compara a tag guardada na linha com a tag procurada
            if cache_validade[linha] == 1 and cache_tag[linha] == tag:
                # ----- ACERTO (HIT) -----
                total_acertos = total_acertos + 1
                if byte == 0:
                    dado = cache_byte0[linha]
                else:
                    dado = cache_byte1[linha]
                print("Resultado: ACERTO (hit) -> dado veio direto da cache:", dado)
            else:
                # ----- FALTA (MISS) -----
                total_faltas = total_faltas + 1
                if cache_validade[linha] == 1 and cache_tag[linha] != tag:
                    # a linha estava ocupada por outro bloco -> conflito
                    total_substituicoes = total_substituicoes + 1
                    print("Resultado: FALTA (miss) por CONFLITO -> a linha", linha,
                          "guardava a tag", cache_tag[linha],
                          "e sera substituida pela tag", tag, ".")
                else:
                    # a linha estava vazia -> primeira vez (compulsorio)
                    print("Resultado: FALTA (miss) compulsoria -> a linha", linha, "estava vazia.")
                # busca o bloco inteiro (2 bytes) na RAM e copia para a cache
                endereco_par = bloco * 2          # primeiro endereco do bloco
                cache_validade[linha] = 1
                cache_tag[linha] = tag
                cache_byte0[linha] = ram[endereco_par]
                cache_byte1[linha] = ram[endereco_par + 1]
                # agora entrega o byte que foi pedido
                if byte == 0:
                    dado = cache_byte0[linha]
                else:
                    dado = cache_byte1[linha]
                print("Bloco copiado da RAM para a linha", linha, "-> dado entregue:", dado)
            print()

    # ----------------------------------------------------------------
    # OPCAO 3: LISTAR A MEMORIA RAM
    # ----------------------------------------------------------------
    elif opcao == 3:
        print("--- LISTA DA RAM (32 enderecos, palavras de 8 bits) ---")
        endereco_atual = 0
        while endereco_atual < 32:
            valor_ram = ram[endereco_atual]
            # converte o valor (0 a 255) para 8 bits, do bit 7 ao bit 0
            valor_bin = ""
            posicao = 7
            while posicao >= 0:
                bit = (valor_ram // (2 ** posicao)) % 2
                valor_bin = valor_bin + str(bit)
                posicao = posicao - 1
            # converte o endereco (0 a 31) para 5 bits
            endereco_bin = ""
            posicao = 4
            while posicao >= 0:
                bit = (endereco_atual // (2 ** posicao)) % 2
                endereco_bin = endereco_bin + str(bit)
                posicao = posicao - 1
            print("Endereco", endereco_bin, "(", endereco_atual, ") -> dado", valor_bin, "(", valor_ram, ")")
            endereco_atual = endereco_atual + 1
        print()

    # ----------------------------------------------------------------
    # OPCAO 4: MOSTRAR A CACHE (validade, tag e os dois bytes)
    # ----------------------------------------------------------------
    elif opcao == 4:
        print("--- ESTADO DA CACHE (4 linhas) ---")
        linha_atual = 0
        while linha_atual < 4:
            if cache_validade[linha_atual] == 1:
                # converte os dois bytes guardados para 8 bits
                b0 = cache_byte0[linha_atual]
                b1 = cache_byte1[linha_atual]
                b0_bin = ""
                posicao = 7
                while posicao >= 0:
                    bit = (b0 // (2 ** posicao)) % 2
                    b0_bin = b0_bin + str(bit)
                    posicao = posicao - 1
                b1_bin = ""
                posicao = 7
                while posicao >= 0:
                    bit = (b1 // (2 ** posicao)) % 2
                    b1_bin = b1_bin + str(bit)
                    posicao = posicao - 1
                # tag em 2 bits
                t = cache_tag[linha_atual]
                tag_bin = str(t // 2) + str(t % 2)
                print("Linha", linha_atual, "-> valida: 1 | tag:", tag_bin,
                      "| byte0:", b0_bin, "(", b0, ") | byte1:", b1_bin, "(", b1, ")")
            else:
                print("Linha", linha_atual, "-> valida: 0 (vazia)")
            linha_atual = linha_atual + 1
        print()

    # ----------------------------------------------------------------
    # OPCAO 5: ENCERRAR E MOSTRAR ESTATISTICAS
    # ----------------------------------------------------------------
    elif opcao == 5:
        print("--- ENCERRANDO O SIMULADOR ---")
        print("Total de leituras:", total_leituras)
        print("Acertos (hits):", total_acertos)
        print("Faltas (misses):", total_faltas)
        print("Substituicoes por conflito:", total_substituicoes)
        if total_leituras > 0:
            taxa = total_acertos / total_leituras * 100
            print("Taxa de acerto:", round(taxa, 2), "%")

    # ----------------------------------------------------------------
    # qualquer outra opcao
    # ----------------------------------------------------------------
    else:
        print("Opcao desconhecida:", opcao)

--- ESCRITA ---
Gravando o valor 170 no endereco 2
O bloco nao estava na cache -> atualizei apenas a RAM.

--- ESCRITA ---
Gravando o valor 85 no endereco 3
O bloco nao estava na cache -> atualizei apenas a RAM.

--- LEITURA ---
Endereco 2 = 00010 ( TAG 00 | LINHA 01 | BYTE 0 )
Bloco: 1  | Linha: 1  | Tag procurada: 0  | Deslocamento: 0
Resultado: FALTA (miss) compulsoria -> a linha 1 estava vazia.
Bloco copiado da RAM para a linha 1 -> dado entregue: 170

--- LEITURA ---
Endereco 2 = 00010 ( TAG 00 | LINHA 01 | BYTE 0 )
Bloco: 1  | Linha: 1  | Tag procurada: 0  | Deslocamento: 0
Resultado: ACERTO (hit) -> dado veio direto da cache: 170

--- LEITURA ---
Endereco 3 = 00011 ( TAG 00 | LINHA 01 | BYTE 1 )
Bloco: 1  | Linha: 1  | Tag procurada: 0  | Deslocamento: 1
Resultado: ACERTO (hit) -> dado veio direto da cache: 85

--- LEITURA ---
Endereco 10 = 01010 ( TAG 01 | LINHA 01 | BYTE 0 )
Bloco: 5  | Linha: 1  | Tag procurada: 1  | Deslocamento: 0
Resultado: FALTA (miss) por CONFLITO -> a l

<a id="a27-testes"></a>
## Testes

A tabela abaixo relaciona cada comando do roteiro com o comportamento esperado. Basta
comparar com a saída do programa principal (impressa acima).

| # | Comando | Endereço (5 bits) | Esperado |
|---|---------|-------------------|----------|
| 1 | escrever 170 | end. 2 = `00010` | grava na RAM (bloco ainda fora da cache) |
| 2 | escrever 85  | end. 3 = `00011` | grava na RAM (bloco ainda fora da cache) |
| 3 | ler 2 | `00010` (T 00, L 01, B 0) | **MISS compulsório** (linha 1 vazia) |
| 4 | ler 2 | `00010` | **HIT** |
| 5 | ler 3 | `00011` (T 00, L 01, B 1) | **HIT** (outro byte do mesmo bloco) |
| 6 | ler 10 | `01010` (T 01, L 01, B 0) | **MISS por conflito** → substituição na linha 1 |
| 7 | ler 2 | `00010` | **MISS por conflito** (o bloco antigo foi expulso) |
| 8 | ler 0 | `00000` | **MISS** — limite inferior do endereço |
| 9 | ler 31 | `11111` (T 11, L 11, B 1) | **MISS** — limite superior do endereço |
| 10 | escrever 200 | end. 2 | bloco está na cache → atualiza RAM **e** cache |
| 11 | escrever 123 | end. 10 | bloco fora da cache → atualiza **só** a RAM |

Os quatro casos centrais pedidos no enunciado aparecem nas linhas 3, 4, 6 e 7:
primeira leitura → *miss*; segunda leitura do mesmo bloco → *hit*; acesso a outro bloco
na mesma linha → *substituição*; novo acesso ao bloco antigo → *miss por conflito*.


### Validação por um segundo método (conferência das fórmulas)

Para não confiar só no programa, a célula abaixo recalcula **tag, linha e byte** de cada
endereço de teste por **dois caminhos independentes** e confere se dão o mesmo resultado:

- **Caminho A (aritmético):** com as divisões e restos das fórmulas.
- **Caminho B (fatiando os bits):** montando o endereço em 5 bits e separando os pedaços.


In [5]:
# enderecos usados nos testes
enderecos_teste = [0, 2, 3, 10, 31]

print("end | binario | TAG LINHA BYTE (A=aritmetica  B=bits) | confere?")
for endereco in enderecos_teste:
    # ---- Caminho A: aritmetica ----
    tag_a = endereco // 8
    linha_a = (endereco // 2) % 4
    byte_a = endereco % 2

    # ---- monta o endereco em 5 bits ----
    endereco_bin = ""
    posicao = 4
    while posicao >= 0:
        bit = (endereco // (2 ** posicao)) % 2
        endereco_bin = endereco_bin + str(bit)
        posicao = posicao - 1

    # ---- Caminho B: separa os bits do endereco ----
    # endereco_bin tem 5 caracteres: posicoes 0,1 = tag; 2,3 = linha; 4 = byte
    tag_bits = endereco_bin[0] + endereco_bin[1]
    linha_bits = endereco_bin[2] + endereco_bin[3]
    byte_bits = endereco_bin[4]
    tag_b = int(tag_bits, 2)      # converte "01" -> 1
    linha_b = int(linha_bits, 2)
    byte_b = int(byte_bits, 2)

    # ---- confere ----
    if tag_a == tag_b and linha_a == linha_b and byte_a == byte_b:
        confere = "OK"
    else:
        confere = "ERRO"

    print(endereco, " | ", endereco_bin, " | A:", tag_a, linha_a, byte_a,
          " B:", tag_b, linha_b, byte_b, " | ", confere)

end | binario | TAG LINHA BYTE (A=aritmetica  B=bits) | confere?
0  |  00000  | A: 0 0 0  B: 0 0 0  |  OK
2  |  00010  | A: 0 1 0  B: 0 1 0  |  OK
3  |  00011  | A: 0 1 1  B: 0 1 1  |  OK
10  |  01010  | A: 1 1 0  B: 1 1 0  |  OK
31  |  11111  | A: 3 3 1  B: 3 3 1  |  OK


<a id="a27-conclusao"></a>
## Conclusão

O simulador reproduz o exemplo do slide 41 com uma memória principal de 32 endereços
(5 bits, palavras de 8 bits), blocos de 2 bytes e uma cache de **mapeamento direto** com
4 linhas. O endereço foi dividido em **TAG (2 bits) + LINHA (2 bits) + BYTE (1 bit)** e o
programa mostra, em cada acesso, a decomposição do endereço, a linha consultada, a
comparação de tags e o resultado **acerto/falta**, sinalizando a **substituição por
conflito** quando duas regiões diferentes disputam a mesma linha.

A sequência de testes confirmou o comportamento esperado do mapeamento direto: a primeira
leitura de um bloco é sempre uma falta compulsória; releituras do mesmo bloco viram
acertos (inclusive ao pedir o outro byte do bloco); e, como cada bloco tem uma linha
fixa, basta acessar um bloco "vizinho" que cai na mesma linha para provocar substituição,
fazendo o bloco anterior virar falta na próxima vez. As estatísticas finais foram
**7 leituras, 2 acertos, 5 faltas, 2 substituições** e taxa de acerto de **28,57%** —
valor baixo justamente porque os testes foram montados para forçar os conflitos.

### Hipóteses e escolhas de teste (declaradas)

- O slide **não especifica o conteúdo** das palavras da RAM nem dos dados da cache (as
  células do desenho estão em branco). Por isso a RAM começa **zerada**, como no
  simulador da Atividade 2.1, e os valores **170, 85, 200 e 123** são **escolhas minhas
  apenas para teste**, não dados atribuídos ao slide.
- **Política de escrita:** escrita direta (*write-through*) **sem alocação na escrita**
  (*no-write-allocate*): toda escrita vai à RAM e a cache só é atualizada se o bloco já
  estiver presente. Foi a política pedida no enunciado e por isso não há bit "sujo"
  (*dirty*) nas linhas.

### Testes realizados e resultado

- miss compulsório, hit, hit no outro byte do bloco, miss por conflito com substituição,
  miss por conflito ao voltar ao bloco antigo, leituras nos limites (endereços 0 e 31),
  escrita com bloco presente (atualiza cache) e escrita com bloco ausente (só RAM):
  **todos passaram**, conforme a tabela de testes.
- A conferência por segundo método (aritmética × separação de bits) retornou **OK** para
  todos os endereços testados.

### Limitações

- Nesta Atividade 2.7, as operações são fornecidas por um **roteiro pré-carregado**
  (`comandos`) para que seus testes sejam reproduzíveis. A Atividade 2.1 preserva o modo
  interativo com `input()`, mas usa exemplos automáticos por padrão.
- O mapeamento direto não tem política de substituição configurável (LRU, FIFO etc.):
  a linha de cada bloco é fixa, então a "substituição" é sempre a troca obrigatória da
  única linha possível.
- O simulador trabalha byte a byte e não modela tempos de acesso nem hierarquia com mais
  níveis de cache; o foco é demonstrar a lógica de tag/linha/byte e de acerto/falta.
